In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
import torch
from datasets import load_dataset
from transformers import DataCollatorForTokenClassification
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from time import perf_counter
from torch.optim import AdamW
import matplotlib.pyplot as plt
from transformers import pipeline
import gc

In [ ]:
device = "cuda" 

In [ ]:
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B", device_map=device)

## Question 2.1

In [ ]:
r16_config = LoraConfig(
                r                       = 16,
                lora_alpha              = 16,
                lora_dropout            = 0.0,
                target_modules          = "all-linear",
                bias                    = "none",
                task_type               = "CAUSAL_LM",
            )

In [ ]:
model_16r = get_peft_model(model, r16_config)

In [ ]:
model_16r.print_trainable_parameters()

In [ ]:
del model_16r
del model

# Question 2.2

In [ ]:
BATCH_SIZE = 1
EPOCHS = 5
LR = 0.0001

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")

In [ ]:
train = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft").select(range(1001)) # have an extra example bc one instance won't have any assistant text with
print(f"Train Dataset Length: {len(train)}")

test = load_dataset("HuggingFaceH4/ultrachat_200k", split="test_sft").select(range(2500))
print(f"Test Dataset Length: {len(test)}")

print(train[0])

In [ ]:
special_tokens = ["<|im_start|>", "user", "<|im_end|>", "assistant", "\n", "system"]

for token in special_tokens:
    print(f"Token {token}: ID: {tokenizer.encode(token)}")

In [ ]:
ASSISTANT_START = [151644, 77091, 198]
END = 151645

In [ ]:
def tokenize(example):
    tokenized = tokenizer.apply_chat_template(example['messages'], tokenize=True, return_dict=True, add_generation_prompt=False, truncation=True, max_length=2048)

    # only train on assistant text
    labels = []
    i = 0
    recording = False
    while i < len(tokenized['input_ids']):
        if tokenized['input_ids'][i:i+3] == ASSISTANT_START:
            recording = True
            i += 3
            labels.extend([-100, -100, -100])
            continue

        if recording:
            labels.append(tokenized['input_ids'][i])
        else:
            labels.append(-100)

        if tokenized['input_ids'][i] == END:
            recording = False

        i += 1

    assert len(labels) == len(tokenized['input_ids'])
    tokenized['labels'] = labels

    return tokenized

In [ ]:
# make sure we have some assistant text to train on
train = train.map(tokenize, batched = False, remove_columns=train.column_names).filter(lambda x: any(label != -100 for label in x["labels"]))
test = test.map(tokenize, batched = False, remove_columns=test.column_names).filter(lambda x: any(label != -100 for label in x["labels"]))

assert len(train) == 1000
assert len(test) == 2500

coallator = DataCollatorForTokenClassification(tokenizer = tokenizer, padding = True, return_tensors = "pt", label_pad_token_id=-100)

train_loader = DataLoader(train , batch_size = BATCH_SIZE, shuffle = True, collate_fn = coallator)
test_loader =  DataLoader(test, batch_size = BATCH_SIZE, shuffle = False, collate_fn = coallator)


In [ ]:
def epoch(train_set, model, optimizer):
    model.train()
    batch_loss = []
    
    for batch in tqdm(train_set, desc="Train Batch: ", leave=False, position=1):
        batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

        optimizer.zero_grad()
                    
        loss = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], labels = batch["labels"]).loss

        if not torch.isfinite(loss):
            raise RuntimeError(f"Got bad loss, exiting. \nTokens: {batch['input_ids']} \nLabels: {batch['labels']}")

        loss.backward()
        optimizer.step()

        batch_loss.append(loss.item())

    return batch_loss


@torch.no_grad()
def evaluate(test_set, model):
    model.eval()
    batch_loss = []

    for batch in tqdm(test_set, desc="Test Batch: ", leave=False, position=1):
        batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

        loss = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], labels = batch["labels"]).loss

        batch_loss.append(loss.item())
        
    return batch_loss


def train_loop(train_set, test_set, model, optimizer, epochs):
    total_time = 0.0
    torch.cuda.reset_peak_memory_stats()

    train_loss_hist = []
    test_loss_hist = []

    for _ in tqdm(range(epochs), desc="Epoch: ", leave=True, position=0):
        torch.cuda.synchronize()
        start = perf_counter()

        train_loss = epoch(train_set, model, optimizer)

        torch.cuda.synchronize()
        elapsed = perf_counter() - start
        total_time += elapsed

        test_loss = evaluate(test_set, model)

        train_loss_hist.append(sum(train_loss)/len(train_loss))
        test_loss_hist.append(sum(test_loss)/len(test_loss))

    peak_allocated = torch.cuda.max_memory_allocated() / 1024**3
    print(f"Peak allocated GPU memory: {peak_allocated:.4f} GB")

    print(f"Training time: {total_time:.4f} seconds")

    return train_loss_hist, test_loss_hist

### Full Fine Tune

In [ ]:
model_full = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B", device_map=device)
optimizer = AdamW(params=[param for param in model_full.parameters() if param.requires_grad], lr=LR)

In [ ]:
train_loss_hist, test_loss_hist = train_loop(train_loader, test_loader, model_full, optimizer, EPOCHS)

In [ ]:
model_full.save_pretrained("./full_ft")
tokenizer.save_pretrained("./full_ft")

In [ ]:
del model_full
del optimizer
gc.collect()

In [ ]:
epochs = range(1, len(train_loss_hist) + 1)

plt.figure(figsize=(8, 5))

plt.plot(epochs, train_loss_hist, marker="o", label="Train Loss")
plt.plot(epochs, test_loss_hist, marker="o", label="Test Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Test Loss (Full Model)")
plt.legend()
plt.grid(True)

plt.show()

### LoRA R=1

In [ ]:
r1_config = LoraConfig(
                r                       = 1,
                lora_alpha              = 1,
                lora_dropout            = 0.0,
                target_modules          = "all-linear",
                bias                    = "none",
                task_type               = "CAUSAL_LM",
            )

In [ ]:
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B", device_map=device)
model_1r = get_peft_model(model, r1_config)

In [ ]:
model_1r.print_trainable_parameters()

In [ ]:
optimizer = AdamW(params=[param for param in model_1r.parameters() if param.requires_grad], lr=LR)

In [ ]:
train_loss_hist, test_loss_hist = train_loop(train_loader, test_loader, model_1r, optimizer, EPOCHS)
model_1r = model_1r.merge_and_unload()

model_1r.save_pretrained("./1r")
tokenizer.save_pretrained("./1r")

In [ ]:
del model_1r
del optimizer
del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
epochs = range(1, len(train_loss_hist) + 1)

plt.figure(figsize=(8, 5))

plt.plot(epochs, train_loss_hist, marker="o", label="Train Loss")
plt.plot(epochs, test_loss_hist, marker="o", label="Test Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Test Loss (r=1 LoRA)")
plt.legend()
plt.grid(True)

plt.show()

### LoRA R=4

In [ ]:
r4_config = LoraConfig(
                r                       = 4,
                lora_alpha              = 4,
                lora_dropout            = 0.0,
                target_modules          = "all-linear",
                bias                    = "none",
                task_type               = "CAUSAL_LM",
            )

In [ ]:
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B", device_map=device)
model_4r = get_peft_model(model, r4_config)

In [ ]:
model_4r.print_trainable_parameters()

In [ ]:
optimizer = AdamW(params=[param for param in model_4r.parameters() if param.requires_grad], lr=LR)

In [ ]:
train_loss_hist, test_loss_hist = train_loop(train_loader, test_loader, model_4r, optimizer, EPOCHS)
model_4r = model_4r.merge_and_unload()

model_4r.save_pretrained("./4r")
tokenizer.save_pretrained("./4r")

In [ ]:
del model_4r
del optimizer
del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
epochs = range(1, len(train_loss_hist) + 1)

plt.figure(figsize=(8, 5))

plt.plot(epochs, train_loss_hist, marker="o", label="Train Loss")
plt.plot(epochs, test_loss_hist, marker="o", label="Test Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Test Loss (r=4 LoRA)")
plt.legend()
plt.grid(True)

plt.show()

### LoRA R=16

In [ ]:
r16_config = LoraConfig(
                r                       = 16,
                lora_alpha              = 16,
                lora_dropout            = 0.0,
                target_modules          = "all-linear",
                bias                    = "none",
                task_type               = "CAUSAL_LM",
            )

In [ ]:
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B", device_map=device)
model_16r = get_peft_model(model, r16_config)

In [ ]:
model_16r.print_trainable_parameters()

In [ ]:
optimizer = AdamW(params=[param for param in model_16r.parameters() if param.requires_grad], lr=LR)

In [ ]:
train_loss_hist, test_loss_hist = train_loop(train_loader, test_loader, model_16r, optimizer, EPOCHS)
model_16r = model_16r.merge_and_unload()

model_16r.save_pretrained("./16r")
tokenizer.save_pretrained("./16r")

In [ ]:
del model_16r
del optimizer
del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
epochs = range(1, len(train_loss_hist) + 1)

plt.figure(figsize=(8, 5))

plt.plot(epochs, train_loss_hist, marker="o", label="Train Loss")
plt.plot(epochs, test_loss_hist, marker="o", label="Test Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Test Loss (r=16 LoRA)")
plt.legend()
plt.grid(True)

plt.show()

## Qualitative Evaluation

In [ ]:
prompts = [
    "I am going on a trip to Maui!! Where should I visit on the island?",
    "I am having a rough day :(. What can I do to make myself feel better?",
    "I'm looking to start weight lifting. What should I do to get started?"
]

In [ ]:
def evaluate(save_path):
    pipe = pipeline("text-generation", model=save_path, tokenizer=save_path, device=device)

    outputs = pipe(prompts, max_new_tokens=128, do_sample=False)

    for prompt, output in zip(prompts, outputs):
        print(f"PROMPT: {prompt}")
        print("\n")
        print(f"RESPONSE: {output[0]["generated_text"]}")
        print("****************************")

In [ ]:
evaluate("./full_ft")

In [ ]:
evaluate("./1r")

In [ ]:
evaluate("./4r")

In [ ]:
evaluate("./16r")